Implement and train Softmax Regression with mini-batch SGD and early stopping.

The expected outcome.
* Implement Softmax Regression Model.
* Implement mini-batch SGD.
* The training should support early stopping.
* Train and evaluate the model with cross-validation. The evaluation metric is the *accuracy*.
* Retrain the model with early stopping.


**DO NOT USE SKLEARN**

In [39]:
import numpy as np
import pandas as pd 

from sklearn import datasets
from sklearn.model_selection import StratifiedShuffleSplit

np.random.seed(42)

In [40]:

iris = datasets.load_iris()
X = iris["data"]
y = iris["target"]
df = pd.DataFrame({fname: values for fname, values in zip(iris["feature_names"], X.T)})
df["target"] = y

df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


## Your Code
You can start writing your code from here. Please don't modify any of the previous code.

In [ ]:
class Softmaxregression:
    """
    this part is mainly for me to remember specific parts later on
    minibatchSGD with early stopping
    axis matter when dealing with vectorized operations that has multiple samples
    when calculating valid loss add the + 1e-9 to avoid log(0) and mean to make it scaler value
    in softmax equation making data exponentially then minus it from max eponential wont change the percentage result yet it will avoid overflow issues because exponential grows very fast
    """
    def __init__(self, lr=0.01, epochs=1000):
        self.lr = lr
        self.epochs = epochs
        self.weights = None
        self.biases = None
    
    def softmax(self, z):
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)
    
    def MB_SGD(self, X, y, X_val=None, y_val=None, batch_size=16, patience=5):
        n_samples = X.shape[0]
        indices = np.arange(n_samples)
        np.random.shuffle(indices)
        best_val_loss = np.inf
        epochs_no_improve = 0
        for j in range(self.epochs):
            for i in range(0, n_samples, batch_size):
                batch_indices = indices[i:i+batch_size]
                X_batch = X[batch_indices]
                y_batch = y[batch_indices]
                scores = np.dot(X_batch, self.weights) + self.biases
                probs = self.softmax(scores)
                assert probs.shape == y_batch.shape, "Shape mismatch between probabilities and labels"
                error = probs - y_batch
                dw = np.dot(X_batch.T, error) / len(X_batch)
                db = np.sum(error, axis=0) / len(X_batch)
                self.weights -= self.lr * dw
                self.biases -= self.lr * db

            if X_val is not None and y_val is not None:
                val_scores = np.dot(X_val, self.weights) + self.biases
                val_probs = self.softmax(val_scores)
                val_loss = -np.mean(np.sum(y_val * np.log(val_probs + 1e-9), axis=1)) 
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    print(f"Early stopping at epoch {j+1}\nValidation Loss: {best_val_loss:.4f}\n")
                    break
                else:
                    print(f"Epoch {j+1}, Validation Loss: {val_loss:.4f}")

    
    def fit(self, X, y, X_val=None, y_val=None, patience=5):
        n_features = X.shape[1]
        n_classes = len(np.unique(y))
        self.weights = np.random.randn(n_features, n_classes)
        self.biases = np.random.randn(n_classes)
        y_encoded = pd.get_dummies(y, dtype=int).values
        if y_val is not None:
            y_val_encoded = pd.get_dummies(y_val, dtype=int).values
        else:
            y_val_encoded = None           
        self.MB_SGD(X, y_encoded, X_val, y_val_encoded, patience=patience)

    def predict(self, X):
        scores = np.dot(X, self.weights) + self.biases
        probs = self.softmax(scores)
        return np.argmax(probs, axis=1)

    def score(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == y)
    

In [56]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
softmax = Softmaxregression(epochs=100000)
softmax.fit(X_train, y_train, X_val, y_val, patience=4)

Epoch 1, Validation Loss: 8.5371
Epoch 2, Validation Loss: 5.4150
Epoch 3, Validation Loss: 3.0956
Epoch 4, Validation Loss: 1.9606
Epoch 5, Validation Loss: 1.5672
Epoch 6, Validation Loss: 1.4604
Epoch 7, Validation Loss: 1.3776
Epoch 8, Validation Loss: 1.3005
Epoch 9, Validation Loss: 1.2285
Epoch 10, Validation Loss: 1.1617
Epoch 11, Validation Loss: 1.1002
Epoch 12, Validation Loss: 1.0439
Epoch 13, Validation Loss: 0.9928
Epoch 14, Validation Loss: 0.9466
Epoch 15, Validation Loss: 0.9049
Epoch 16, Validation Loss: 0.8674
Epoch 17, Validation Loss: 0.8337
Epoch 18, Validation Loss: 0.8034
Epoch 19, Validation Loss: 0.7762
Epoch 20, Validation Loss: 0.7516
Epoch 21, Validation Loss: 0.7295
Epoch 22, Validation Loss: 0.7094
Epoch 23, Validation Loss: 0.6912
Epoch 24, Validation Loss: 0.6747
Epoch 25, Validation Loss: 0.6596
Epoch 26, Validation Loss: 0.6458
Epoch 27, Validation Loss: 0.6331
Epoch 28, Validation Loss: 0.6214
Epoch 29, Validation Loss: 0.6106
Epoch 30, Validation Lo

## i have no idea whats wrong with my early stopping implementation
## for 1 million epochs even the valid loss is similar for multiple epochs in a row it doesnt break out of training loop

In [67]:
softmax = Softmaxregression(lr=0.1, epochs=100000)
softmax.fit(X_train, y_train, X_val, y_val, patience=4)

Epoch 1, Validation Loss: 1.0681
Epoch 2, Validation Loss: 0.8249
Epoch 3, Validation Loss: 0.7252
Epoch 4, Validation Loss: 0.6692
Epoch 5, Validation Loss: 0.6303
Epoch 6, Validation Loss: 0.6003
Epoch 7, Validation Loss: 0.5754
Epoch 8, Validation Loss: 0.5539
Epoch 9, Validation Loss: 0.5348
Epoch 10, Validation Loss: 0.5173
Epoch 11, Validation Loss: 0.5012
Epoch 12, Validation Loss: 0.4859
Epoch 13, Validation Loss: 0.4715
Epoch 14, Validation Loss: 0.4575
Epoch 15, Validation Loss: 0.4440
Epoch 16, Validation Loss: 0.4308
Epoch 17, Validation Loss: 0.4178
Epoch 18, Validation Loss: 0.4050
Epoch 19, Validation Loss: 0.3923
Epoch 20, Validation Loss: 0.3797
Epoch 21, Validation Loss: 0.3675
Epoch 22, Validation Loss: 0.3556
Epoch 23, Validation Loss: 0.3444
Epoch 24, Validation Loss: 0.3340
Epoch 25, Validation Loss: 0.3244
Epoch 26, Validation Loss: 0.3157
Epoch 27, Validation Loss: 0.3081
Epoch 28, Validation Loss: 0.3013
Epoch 29, Validation Loss: 0.2953
Epoch 30, Validation Lo

## it worked when i increased learning rate, i think issue has to be related with float points but im not sure.
## Iam leaving this markdown for explaination when the time comes

In [65]:
softmax = Softmaxregression(lr=0.3, epochs=1000)
softmax.fit(X_train, y_train, X_val, y_val, patience=4)

Epoch 1, Validation Loss: 3.8729
Epoch 2, Validation Loss: 3.7550
Epoch 3, Validation Loss: 2.0261
Epoch 4, Validation Loss: 3.3332
Epoch 5, Validation Loss: 2.4904
Epoch 6, Validation Loss: 2.6834
Early stopping at epoch 7
Validation Loss: 2.0261



In [66]:
softmax.fit(X_train, y_train)
softmax.score(X_train, y_train) , softmax.score(X_val, y_val)

(np.float64(0.9916666666666667), np.float64(0.9666666666666667))

## testing stuff

In [60]:
x = [1,2,3]
exp_X = np.exp(x)
print(exp_X, exp_X/np.sum(exp_X))
y = [[1,2,3],
     [4,5,6]]
exp_y = np.exp(y)
print("\n")
print(exp_y, exp_y/np.sum(exp_y))
print("\n")
print((exp_y/np.sum(exp_y, axis=1, keepdims=True)))

[ 2.71828183  7.3890561  20.08553692] [0.09003057 0.24472847 0.66524096]


[[  2.71828183   7.3890561   20.08553692]
 [ 54.59815003 148.4131591  403.42879349]] [[0.00426978 0.01160646 0.03154963]
 [0.08576079 0.23312201 0.63369132]]


[[0.09003057 0.24472847 0.66524096]
 [0.09003057 0.24472847 0.66524096]]


In [61]:
y = [0,1,2,1,0,2,1,0]
y = np.array(y)
y_encoded = pd.get_dummies(y, dtype=int).values
y_encoded

array([[1, 0, 0],
       [0, 1, 0],
       [0, 0, 1],
       [0, 1, 0],
       [1, 0, 0],
       [0, 0, 1],
       [0, 1, 0],
       [1, 0, 0]])

In [62]:
y = [1,2,3,4,5]
len(np.unique(y))

5

Using the following cell to train and evaluate your model.

In [63]:
split = StratifiedShuffleSplit(n_splits=3, test_size=0.2, random_state=42)
for train_index, test_index in split.split(df, df["target"]):
    strat_train_set = df.loc[train_index]
    strat_test_set = df.loc[test_index]
    softmax = Softmaxregression()
    X_train = strat_train_set.drop("target", axis=1).values
    y_train = strat_train_set["target"].values
    X_test = strat_test_set.drop("target", axis=1).values
    y_test = strat_test_set["target"].values
    softmax.fit(X_train, y_train, X_test, y_test, patience=1)
    
    # Use strat_train_set and strat_test_set to train and evaluate your model

Epoch 1, Validation Loss: 1.6290
Epoch 2, Validation Loss: 1.1299
Epoch 3, Validation Loss: 0.8173
Epoch 4, Validation Loss: 0.6835
Epoch 5, Validation Loss: 0.6428
Epoch 6, Validation Loss: 0.6286
Epoch 7, Validation Loss: 0.6196
Epoch 8, Validation Loss: 0.6118
Epoch 9, Validation Loss: 0.6043
Epoch 10, Validation Loss: 0.5973
Epoch 11, Validation Loss: 0.5906
Epoch 12, Validation Loss: 0.5843
Epoch 13, Validation Loss: 0.5783
Epoch 14, Validation Loss: 0.5726
Epoch 15, Validation Loss: 0.5672
Epoch 16, Validation Loss: 0.5621
Epoch 17, Validation Loss: 0.5572
Epoch 18, Validation Loss: 0.5525
Epoch 19, Validation Loss: 0.5480
Epoch 20, Validation Loss: 0.5437
Epoch 21, Validation Loss: 0.5395
Epoch 22, Validation Loss: 0.5355
Epoch 23, Validation Loss: 0.5316
Epoch 24, Validation Loss: 0.5279
Epoch 25, Validation Loss: 0.5243
Epoch 26, Validation Loss: 0.5208
Epoch 27, Validation Loss: 0.5174
Epoch 28, Validation Loss: 0.5141
Epoch 29, Validation Loss: 0.5109
Epoch 30, Validation Lo